!! POT SER QUE FALLI SI L'ORDINADOR NO TÉ PROU MEMÒRIA RAM

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils.metrics import compute_metrics
from utils.mused import get_span_annotations
from utils.regressions import (
    compare_zscore_permutation,
    compute_all_regressions,
    compute_fisher_overlap,
    compute_participant_baselines,
    compute_permutation_pvalues,
    compute_span_coverage,
    compute_span_hotspots,
    compute_zscores,
    get_top_hotspots_per_text,
    precompute_aoi_arrays,
    run_permutation_test,
)
from utils.tobii import compute_all_token_metrics, get_tfds, read_all_data

pd.set_option("display.max_columns", None)

DATA = "../data/"
TOBII = DATA + "tobii/"

In [ ]:
mused_chosen = pd.read_csv(DATA + "mused_chosen_data.csv")
mused_chosen["num_id"] = mused_chosen.id.str.split("_").str[1].astype(int)
texts = dict(zip(mused_chosen.num_id, mused_chosen.text_clean))
text_ids = mused_chosen.num_id.tolist()

In [ ]:
general = (
    pd.read_csv(TOBII + "general.tsv", sep="\t")
    .drop(["Event", "Recording timestamp", "Computer timestamp"], axis=1)
    .drop_duplicates()
)

In [ ]:
aoi_hit, calibration_dfs, all_participants, dfs = read_all_data(TOBII + "all_parquets")
participants = list(dict.fromkeys([x[0] for x in all_participants]))
text_dfs, aoi_cols_dict, tfds = get_tfds(dfs, aoi_hit, participants, text_ids)

# Anonymize participant names: M1-M5 (male), F1-F5 (female)
general = pd.read_csv(TOBII + "general.tsv", sep="\t")[
    ["Participant name", "sexe"]
].drop_duplicates()
gender_series = general.set_index("Participant name")["sexe"]


# Build lookup that handles name mismatches (e.g. Mireia_alvarez vs mireia)
def lookup_gender(name):
    if name in gender_series.index:
        return gender_series[name]
    lower = name.lower().split("_")[0]
    for gs_name, gs_gender in gender_series.items():
        if gs_name.lower() == lower:
            return gs_gender
    return None


males = [p for p in participants if lookup_gender(p) == "home"]
females = [p for p in participants if lookup_gender(p) == "dona"]
name_map = {name: f"M{i + 1}" for i, name in enumerate(males)}
name_map.update({name: f"F{i + 1}" for i, name in enumerate(females)})
gender_map = {name_map[k]: lookup_gender(k) for k in name_map}
print("Participant mapping:")
for real, anon in name_map.items():
    print(f"  {real} → {anon} ({lookup_gender(real)})")


# Apply mapping to all DataFrames
def rename_participant(df):
    if "participant" in df.columns:
        df["participant"] = df["participant"].map(name_map).fillna(df["participant"])
    return df


for p_old in list(text_dfs.keys()):
    p_new = name_map.get(p_old, p_old)
    if p_new != p_old:
        text_dfs[p_new] = text_dfs.pop(p_old)
        aoi_cols_dict[p_new] = aoi_cols_dict.pop(p_old)
        tfds[p_new] = tfds.pop(p_old)

participants = list(text_dfs.keys())

# MuSeD spans

In [ ]:
data = pd.read_csv(DATA + "chosen_data_full.csv")
data["label_clean"] = data["label_clean"].apply(lambda x: eval(x))
annotations, annotations_sum, per_label_annotations = get_span_annotations(data)

# Comparison: eye-tracking vs. span annotations

Aquí només es mira total fixation duration (TFD), al final es compara amb FFD (first fixation duration) i FC (fixation count)

In [ ]:
annotated_texts = [
    tid for tid, v in annotations.items() if (v.mean() < 1.0 and v.mean() > 0)
]
print(f"{len(annotated_texts)} texts with partial annotations (out of {len(annotations)})")
print(f"Excluded (all annotated): {sorted(set(annotations) - set(annotated_texts))}")

In [ ]:
metrics_records = []
for participant in participants:
    for text_id in annotated_texts:
        if text_id not in tfds.get(participant, {}):
            continue
        m = compute_metrics(tfds[participant][text_id], annotations_sum[text_id])
        if m is not None:
            metrics_records.append({"participant": participant, "text_id": text_id, **m})

metrics_df = pd.DataFrame(metrics_records)
print(f"Computed {len(metrics_df)} (participant, text) pairs")

In [ ]:
print("=== Spearman correlation (eye-tracking vs annotations, per participant) ===")
print(metrics_df.groupby("participant")["spearman"].agg(["mean", "std", "count"]))
print()
print("=== Cross-entropy H(annotation | eye-tracking, per participant) ===")
print(metrics_df.groupby("participant")["cross_entropy"].agg(["mean", "std", "count"]))
print()
print("=== KL-divergence KL(annotation || eye-tracking, per participant) ===")
print(metrics_df.groupby("participant")["kl_div"].agg(["mean", "std", "count"]))
print("=== JS-divergence JS(annotation || eye-tracking, per participant) ===")
print(metrics_df.groupby("participant")["js_div"].agg(["mean", "std", "count"]))

In [ ]:
text_avg = metrics_df.groupby("text_id").agg(
    spearman_mean=("spearman", "mean"),
    spearman_std=("spearman", "std"),
    ce_mean=("cross_entropy", "mean"),
    ce_std=("cross_entropy", "std"),
    kl_mean=("kl_div", "mean"),
    kl_std=("kl_div", "std"),
    js_mean=("js_div", "mean"),
    js_std=("js_div", "std"),
    n_participants=("participant", "count"),
)
print("=== Per-text metrics (averaged across participants) ===")
print(text_avg.sort_values("spearman_mean", ascending=False).round(2).to_string())
print()
print(f"Overall mean Spearman rho: {text_avg['spearman_mean'].mean():.2f}")
print(f"Overall mean cross-entropy: {text_avg['ce_mean'].mean():.2f}")
print(f"Overall mean KL-divergence: {text_avg['kl_mean'].mean():.2f}")
print(f"Overall mean JS-divergence: {text_avg['js_mean'].mean():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(range(len(text_avg)), text_avg["spearman_mean"])
axes[0].set_xlabel("Text ID")
axes[0].set_ylabel("Mean Spearman rho")
axes[0].set_title("Spearman correlation per text (avg across participants)")
axes[0].axhline(0, color="gray", linestyle="--", alpha=0.5)

axes[1].bar(range(len(text_avg)), text_avg["ce_mean"])
axes[1].set_xlabel("Text ID")
axes[1].set_ylabel("Mean cross-entropy")
axes[1].set_title("Cross-entropy per text (avg across participants)")

axes[2].bar(range(len(text_avg)), text_avg["js_mean"])
axes[2].set_xlabel("Text ID")
axes[2].set_ylabel("Mean JS-divergence")
axes[2].set_title("JS-divergence per text (avg across participants)")

plt.tight_layout()
plt.show()

In [ ]:
part_summary = metrics_df.groupby("participant").agg(
    spearman_mean=("spearman", "mean"),
    ce_mean=("cross_entropy", "mean"),
    kl_mean=("kl_div", "mean"),
    js_mean=("js_div", "mean"),
)
part_summary["sex"] = part_summary.index.map(gender_map)
part_summary = part_summary.sort_values("spearman_mean", ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ["#1f77b4" if s == "home" else "#e377c2" for s in part_summary["sex"]]

axes[0].barh(part_summary.index, part_summary["spearman_mean"], color=colors)
axes[0].set_xlabel("Mean Spearman rho")
axes[0].set_title("Spearman correlation per participant")
axes[0].axvline(0, color="gray", linestyle="--", alpha=0.5)

axes[1].barh(part_summary.index, part_summary["ce_mean"], color=colors)
axes[1].set_xlabel("Mean cross-entropy")
axes[1].set_title("Cross-entropy per participant")

axes[2].barh(part_summary.index, part_summary["js_mean"], color=colors)
axes[2].set_xlabel("Mean JS-divergence")
axes[2].set_title("JS-divergence per participant")

plt.tight_layout()
plt.show()

# Per-label span analysis

In [ ]:
label_metrics_records = []
for tag, tag_anns in per_label_annotations.items():
    for participant in participants:
        for text_id in annotated_texts:
            if text_id not in tfds.get(participant, {}):
                continue
            if text_id not in tag_anns:
                continue
            m = compute_metrics(tfds[participant][text_id], tag_anns[text_id])
            if m is not None:
                label_metrics_records.append(
                    {"label": tag, "participant": participant, "text_id": text_id, **m}
                )

label_metrics_df = pd.DataFrame(label_metrics_records)
print(
    f"Computed {len(label_metrics_df)} records across {label_metrics_df['label'].nunique()} label types"
)

In [ ]:
print("=== Metrics per span label type ===")
print(
    label_metrics_df.groupby("label")
    .agg(
        spearman_mean=("spearman", "mean"),
        spearman_std=("spearman", "std"),
        ce_mean=("cross_entropy", "mean"),
        kl_mean=("kl_div", "mean"),
        js_mean=("js_div", "mean"),
        n=("text_id", "count"),
    )
    .sort_values("spearman_mean", ascending=False)
    .round(2)
    .to_string()
)

In [ ]:
label_summary = (
    label_metrics_df.groupby("label")
    .agg(spearman=("spearman", "mean"), ce=("cross_entropy", "mean"), js=("js_div", "mean"))
    .sort_values("spearman", ascending=True)
)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
axes[0].barh(label_summary.index, label_summary["spearman"])
axes[0].set_xlabel("Mean Spearman rho")
axes[0].set_title("Spearman per label type")
axes[0].axvline(0, color="gray", linestyle="--", alpha=0.5)
axes[1].barh(label_summary.index, label_summary["ce"])
axes[1].set_xlabel("Mean cross-entropy")
axes[1].set_title("Cross-entropy per label type")
axes[2].barh(label_summary.index, label_summary["js"])
axes[2].set_xlabel("Mean JS-divergence")
axes[2].set_title("JS-divergence per label type")
plt.tight_layout()
plt.show()

# Per-binary-column analysis (text-level categories)

In [ ]:
label_cols = [
    "sexist",
    "irony",
    "humor",
    "implicit sexism",
    "stereotypes",
    "inequality",
    "discrimination",
    "objectification",
    "critique",
    "reported_sexism",
]

split_metrics_records = []
for col in label_cols:
    for _, row in data.iterrows():
        text_id = row["num_id"]
        group = row[col]
        for participant in participants:
            if text_id not in tfds.get(participant, {}):
                continue
            if text_id not in annotations_sum:
                continue
            m = compute_metrics(tfds[participant][text_id], annotations_sum[text_id])
            if m is not None:
                split_metrics_records.append(
                    {
                        "column": col,
                        "group": int(group),
                        "participant": participant,
                        "text_id": text_id,
                        **m,
                    }
                )

split_df = pd.DataFrame(split_metrics_records)
print(f"Computed {len(split_df)} records across {split_df['column'].nunique()} columns")

In [ ]:
summary = (
    split_df.groupby(["column", "group"])
    .agg(
        spearman=("spearman", "mean"),
        cross_entropy=("cross_entropy", "mean"),
        js_div=("js_div", "mean"),
        n_texts=("text_id", "nunique"),
        n_pairs=("text_id", "count"),
    )
    .unstack(level="group")
    .round(2)
)
summary.columns = ["_".join(str(c) for c in col).strip("_") for col in summary.columns]
print("=== Metrics split by text-level binary categories ===")
print(summary.to_string())

In [ ]:
deltas = []
for col in label_cols:
    sub = split_df[split_df["column"] == col]
    pos = sub[sub["group"] == 1]
    neg = sub[sub["group"] == 0]
    if len(pos) > 0 and len(neg) > 0:
        for metric in ["spearman", "cross_entropy", "js_div"]:
            deltas.append(
                {
                    "column": col,
                    "metric": metric,
                    "mean_positive": pos[metric].mean(),
                    "mean_negative": neg[metric].mean(),
                    "delta": pos[metric].mean() - neg[metric].mean(),
                }
            )

delta_df = pd.DataFrame(deltas)
print("Delta (positive group - negative group) per category:")
print(delta_df.pivot(index="column", columns="metric", values="delta").round(2).to_string())

In [ ]:
pivot = delta_df.pivot(index="column", columns="metric", values="delta")
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, metric in zip(axes, ["spearman", "cross_entropy", "js_div"]):
    vals = pivot[metric].sort_values()
    colors = ["#2ca02c" if v > 0 else "#d62728" for v in vals]
    ax.barh(vals.index, vals, color=colors)
    ax.set_xlabel(f"Delta {metric}")
    ax.set_title(f"Delta {metric} (has label - no label)")
    ax.axvline(0, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# Regression analysis

In [ ]:
all_regressions_df, all_reg_metrics_df = compute_all_regressions(text_dfs, text_ids)

print(f"Total regressions: {len(all_regressions_df)}")
print(
    f"Regressions per text (mean ± std): {all_reg_metrics_df.groupby('text_id')['total_regressions'].sum().mean():.1f} ± {all_reg_metrics_df.groupby('text_id')['total_regressions'].sum().std():.1f}"
)
print("\nRegressions by type:")
print(all_regressions_df["regression_type"].value_counts())

In [ ]:
participant_baselines, mean_fd_all = compute_participant_baselines(
    text_dfs, text_ids, texts, all_reg_metrics_df, all_regressions_df
)

print(f"Mean fixation duration overall: {mean_fd_all:.1f} ms")
print("Per-participant baselines:")
for p, bl in participant_baselines.items():
    speed = "slow" if bl["mean_fd"] > mean_fd_all else "fast"
    print(
        f"  {p}: rate={bl['regression_rate']:.3f}, mean_fd={bl['mean_fd']:.1f}ms ({speed}), n_regressions={bl['total_regressions']}"
    )

In [ ]:
zscore_target_df, salient_target_tokens = compute_zscores(
    text_ids,
    texts,
    participants,
    text_dfs,
    all_regressions_df,
    all_reg_metrics_df,
    participant_baselines,
    mean_fd_all,
    direction="target",
)
print(f"Salient TARGET tokens (BH q < 0.05): {len(salient_target_tokens)}")
print("\nTop 20:")
salient_target_tokens.head(20)

In [ ]:
zscore_source_df, salient_source_tokens = compute_zscores(
    text_ids,
    texts,
    participants,
    text_dfs,
    all_regressions_df,
    all_reg_metrics_df,
    participant_baselines,
    mean_fd_all,
    direction="source",
)
print(f"Salient SOURCE tokens (BH q < 0.05): {len(salient_source_tokens)}")
print("\nTop 20:")
salient_source_tokens.head(20)

In [ ]:
print("=== Salient TARGET tokens per text ===")
print(
    salient_target_tokens.groupby("text_id")
    .agg(n_salient=("token_idx", "count"), tokens=("token", list), max_z=("z_score", "max"))
    .sort_values("max_z", ascending=False)
    .to_string()
)

print("\n=== Salient SOURCE tokens per text ===")
print(
    salient_source_tokens.groupby("text_id")
    .agg(n_salient=("token_idx", "count"), tokens=("token", list), max_z=("z_score", "max"))
    .sort_values("max_z", ascending=False)
    .to_string()
)

In [ ]:
texts_to_show = sorted(set(zscore_target_df["text_id"]))
fig, axes = plt.subplots(1, 2, figsize=(20, max(8, len(texts_to_show) * 0.4)))

for ax, df, title in [
    (axes[0], zscore_target_df, "Target z-score (regressions TO)"),
    (axes[1], zscore_source_df, "Source z-score (regressions FROM)"),
]:
    max_tok = max(len(texts[t].split()) for t in texts_to_show)
    mat = np.full((len(texts_to_show), max_tok), np.nan)
    for row in df.itertuples():
        t_idx = texts_to_show.index(row.text_id)
        mat[t_idx, row.token_idx] = row.z_score
    im = ax.imshow(
        mat, aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3, interpolation="nearest"
    )
    ax.set_yticks(range(len(texts_to_show)))
    ax.set_yticklabels(texts_to_show, fontsize=7)
    ax.set_xlabel("Token position (normalized)")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label="z-score")

plt.tight_layout()
plt.show()

In [ ]:
pre_arrays, text_ids_with_data = precompute_aoi_arrays(text_dfs, text_ids)
print(f"Pre-computed AOI arrays for {len(pre_arrays)} participant × text pairs")

t0 = time.time()
for _ in range(10):
    for text_id in text_ids_with_data:
        for participant in list(text_dfs.keys()):
            key = (participant, text_id)
            if key not in pre_arrays:
                continue
            aoi_idx = pre_arrays[key]
            perm_aoi_idx = aoi_idx[np.random.permutation(len(aoi_idx))]
            _ = np.diff(perm_aoi_idx)
elapsed_per_perm = (time.time() - t0) / 10
print(
    f"One permutation: {elapsed_per_perm:.2f}s → 10000 permutations: ~{elapsed_per_perm * 10000 / 60:.1f} min"
)

In [ ]:
obs_target_counts, obs_source_counts, null_target_counts, null_source_counts = (
    run_permutation_test(text_ids_with_data, texts, text_dfs, pre_arrays, n_perms=10000)
)

In [ ]:
perm_target_qvalues, perm_source_qvalues, perm_sig_targets, perm_sig_sources = (
    compute_permutation_pvalues(
        text_ids_with_data,
        obs_target_counts,
        obs_source_counts,
        null_target_counts,
        null_source_counts,
    )
)
print(f"Permutation-significant TARGET tokens (q < 0.05): {perm_sig_targets}")
print(f"Permutation-significant SOURCE tokens (q < 0.05): {perm_sig_sources}")

In [ ]:
all_sig_tokens, hotspots_df = compare_zscore_permutation(
    salient_target_tokens,
    salient_source_tokens,
    text_ids_with_data,
    perm_target_qvalues,
    perm_source_qvalues,
    zscore_target_df,
    zscore_source_df,
    texts,
)

z_target_sig = set(
    zip(salient_target_tokens["text_id"], salient_target_tokens["token_idx"])
)
perm_target_sig = set()
for text_id in text_ids_with_data:
    for idx, q in enumerate(perm_target_qvalues[text_id]):
        if q < 0.05:
            perm_target_sig.add((text_id, idx))
z_source_sig = set(
    zip(salient_source_tokens["text_id"], salient_source_tokens["token_idx"])
)
perm_source_sig = set()
for text_id in text_ids_with_data:
    for idx, q in enumerate(perm_source_qvalues[text_id]):
        if q < 0.05:
            perm_source_sig.add((text_id, idx))

both_target = z_target_sig & perm_target_sig
both_source = z_source_sig & perm_source_sig
print(
    f"TARGET: z-only={len(z_target_sig - perm_target_sig)}, perm-only={len(perm_target_sig - z_target_sig)}, both={len(both_target)}"
)
print(
    f"SOURCE: z-only={len(z_source_sig - perm_source_sig)}, perm-only={len(perm_source_sig - z_source_sig)}, both={len(both_source)}"
)
print(f"Total unique significant (text, token): {len(all_sig_tokens)}")

# Top 10 lexical hotspots per text (stopwords filtered)
hotspots_top_df, stopword_pct = get_top_hotspots_per_text(
    zscore_target_df,
    zscore_source_df,
    salient_target_tokens,
    salient_source_tokens,
    texts,
    text_ids_with_data,
    perm_target_qvalues=perm_target_qvalues,
    perm_source_qvalues=perm_source_qvalues,
    n_top=10,
)
print(f"\nStopword percentage in significant tokens: {stopword_pct:.1f}%")
print(f"\n=== Top lexical hotspots per text ({len(hotspots_top_df)} total) ===")
for text_id in sorted(text_ids_with_data):
    th = hotspots_top_df[hotspots_top_df["text_id"] == text_id].sort_values(
        "z_score", key=abs, ascending=False
    )
    if len(th) > 0:
        print(f"\nText {text_id}:")
        print(
            th[["direction", "token", "token_idx", "z_score", "is_stopword"]].to_string(
                index=False
            )
        )

In [ ]:
overlap_df, fisher_df = compute_fisher_overlap(all_sig_tokens, annotations, texts)

total_sig_in_span = overlap_df["in_span"].sum()
total_sig = len(overlap_df)
print(
    f"Significant tokens inside annotated spans: {total_sig_in_span}/{total_sig} ({100 * total_sig_in_span / max(total_sig, 1):.1f}%)"
)
print("\nFisher's exact test per text:")
print(fisher_df.to_string(index=False) if len(fisher_df) > 0 else "No results")

if len(fisher_df) > 0:
    sig_texts = fisher_df[fisher_df["p_value"] < 0.05]
    print(f"\nTexts with significant overlap (p < 0.05): {len(sig_texts)}")
    if len(sig_texts) > 0:
        print(
            sig_texts[
                ["text_id", "a_sig_in_span", "b_sig_not_in_span", "odds_ratio", "p_value"]
            ].to_string(index=False)
        )

In [ ]:
coverage_df, spearman_df = compute_span_coverage(
    text_ids, texts, data, per_label_annotations, all_reg_metrics_df, text_dfs, label_cols
)
print("=== Spearman: span coverage vs regression rate ===")
print(spearman_df.to_string(index=False))

In [ ]:
span_hotspots_df = compute_span_hotspots(
    per_label_annotations, zscore_target_df, zscore_source_df, texts
)

if len(span_hotspots_df) > 0:
    print("=== Span label hotspots (weighted by 1/span_length) ===")
    print(
        span_hotspots_df.groupby(["label", "direction"])
        .agg(
            n_hotspots=("z_score", "count"),
            mean_z=("z_score", "mean"),
            mean_cluster_size=("cluster_size", "mean"),
            mean_span_length=("span_length", "mean"),
        )
        .round(2)
        .to_string()
    )

    print("\n=== Top 20 hotspots by |z_score| ===")
    top = span_hotspots_df.reindex(
        span_hotspots_df["z_score"].abs().sort_values(ascending=False).index
    ).head(20)
    print(
        top[
            [
                "label",
                "text_id",
                "direction",
                "token",
                "position",
                "z_score",
                "cluster_size",
                "span_length",
            ]
        ].to_string(index=False)
    )
else:
    print("No span hotspots found.")

In [ ]:
labels_with_variation = [
    col
    for col in label_cols
    if col in coverage_df["label"].values
    and coverage_df[coverage_df["label"] == col]["coverage"].std() > 0
]

if len(labels_with_variation) > 0:
    n_labels = len(labels_with_variation)
    n_cols_plot = 3
    n_rows = (n_labels + n_cols_plot - 1) // n_cols_plot
    fig, axes = plt.subplots(n_rows, n_cols_plot, figsize=(5 * n_cols_plot, 4 * n_rows))
    axes = axes.flatten() if n_labels > 1 else [axes]

    for ax, col in zip(axes, labels_with_variation):
        sub = coverage_df[coverage_df["label"] == col]
        rho = spearman_df[spearman_df["label"] == col]["rho"].values
        p = spearman_df[spearman_df["label"] == col]["p"].values
        rho_str = f"{rho[0]:.2f}" if len(rho) > 0 else "N/A"
        p_str = f"{p[0]:.4f}" if len(p) > 0 else "N/A"

        ax.scatter(sub["coverage"], sub["mean_regression_rate"], alpha=0.6, s=40)
        if len(sub) > 2:
            z = np.polyfit(sub["coverage"], sub["mean_regression_rate"], 1)
            x_line = np.linspace(sub["coverage"].min(), sub["coverage"].max(), 100)
            ax.plot(x_line, np.polyval(z, x_line), "r--", alpha=0.5)
        ax.set_xlabel("Span coverage")
        ax.set_ylabel("Regression rate")
        ax.set_title(f"{col}\n\u03c1={rho_str}, p={p_str}")
        ax.axvline(0, color="gray", linestyle=":", alpha=0.3)

    for ax in axes[len(labels_with_variation) :]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.show()
else:
    print("No labels with coverage variation to plot.")

# FFD i FC

In [ ]:
token_metrics_df = compute_all_token_metrics(text_dfs, text_ids)
print(f"Per-token entries: {len(token_metrics_df)}")
print(f"Columns: {list(token_metrics_df.columns)}")

In [ ]:
# TFD, FFD and FC analysis: compare sexist vs non-sexist texts
import re

from scipy.stats import mannwhitneyu

# Ensure num_id exists
if "num_id" not in data.columns:
    data["num_id"] = data["id"].str.replace("video_", "").astype(int)

# Map text_id to binary label
text_label_map = {}
for _, row in data.iterrows():
    tid = int(row["id"].split("_")[1])
    text_label_map[tid] = int(row["sexist"])

# Per-text mean TFD, FFD, FC (averaged across participants and tokens)
metrics_by_text = (
    token_metrics_df.groupby("text_id")
    .agg(
        mean_tfd=("tfd", "mean"),
        mean_ffd=("ffd", "mean"),
        mean_fc=("fc", "mean"),
    )
    .reset_index()
)
metrics_by_text["is_sexist"] = metrics_by_text["text_id"].map(text_label_map)

sexist = metrics_by_text[metrics_by_text["is_sexist"] == 1]
non_sexist = metrics_by_text[metrics_by_text["is_sexist"] == 0]

print("=== Text-level comparison (sexist vs non-sexist) ===")
for metric, label in [("mean_tfd", "TFD"), ("mean_ffd", "FFD"), ("mean_fc", "FC")]:
    u, p = mannwhitneyu(sexist[metric], non_sexist[metric], alternative="two-sided")
    print(
        f"  {label:4s}: sexist={sexist[metric].mean():.4f} (±{sexist[metric].std():.4f}), "
        f"non-sexist={non_sexist[metric].mean():.4f} (±{non_sexist[metric].std():.4f}), p={p:.4f}"
    )

# Per-token z-scores (participant baseline) for all three metrics
zscore_dfs = []
for participant in token_metrics_df["participant"].unique():
    p_data = token_metrics_df[token_metrics_df["participant"] == participant].copy()
    for metric in ["tfd", "ffd", "fc"]:
        mean_val = p_data[metric].mean()
        std_val = p_data[metric].std()
        p_data[f"{metric}_zscore"] = (
            (p_data[metric] - mean_val) / std_val if std_val > 0 else 0.0
        )
    zscore_dfs.append(
        p_data[
            [
                "participant",
                "text_id",
                "AOI",
                "token_idx",
                "tfd",
                "ffd",
                "fc",
                "tfd_zscore",
                "ffd_zscore",
                "fc_zscore",
            ]
        ]
    )

z_df = pd.concat(zscore_dfs, ignore_index=True)

# Get span token ranges per text from label_clean
span_tokens = {}
for text_id in texts.keys():
    row = data[data["num_id"] == text_id]
    span_tokens[text_id] = set()
    if len(row) > 0:
        label_clean = row.iloc[0]["label_clean"]
        labels = eval(label_clean) if isinstance(label_clean, str) else label_clean
        if isinstance(labels, list):
            matches = list(re.finditer(r"\S+", texts[text_id]))
            idx2token = {m.start(): i for i, m in enumerate(matches)}
            for ann in labels:
                s = ann["start"]
                e = ann["end"]
                while idx2token.get(s) is None and s < len(texts[text_id]):
                    s += 1
                while idx2token.get(e) is None and e >= 0:
                    e -= 1
                if idx2token.get(s) is not None and idx2token.get(e) is not None:
                    for i in range(idx2token[s], idx2token[e] + 1):
                        span_tokens[text_id].add(i)

z_df["in_span"] = z_df.apply(
    lambda r: r["token_idx"] in span_tokens.get(r["text_id"], set()), axis=1
)

print("\n=== Token-level z-scores by span membership ===")
for metric, label in [("tfd_zscore", "TFD"), ("ffd_zscore", "FFD"), ("fc_zscore", "FC")]:
    in_s = z_df[z_df["in_span"]][metric].dropna()
    out_s = z_df[~z_df["in_span"]][metric].dropna()
    if len(in_s) > 0 and len(out_s) > 0:
        u, p = mannwhitneyu(in_s, out_s, alternative="two-sided")
        print(
            f"  {label:4s}: inside={in_s.mean():.3f} (n={len(in_s)}), outside={out_s.mean():.3f} (n={len(out_s)}), p={p:.4f}"
        )

# Correlation between the three metrics
print("\n=== Spearman correlation between metrics (per token) ===")
from scipy.stats import spearmanr

for m1, m2 in [("tfd", "ffd"), ("tfd", "fc"), ("ffd", "fc")]:
    rho, p = spearmanr(z_df[m1], z_df[m2])
    print(f"  {m1.upper()} vs {m2.upper()}: rho={rho:.3f}, p={p:.4f}")